In [ ]:
from ultralytics import YOLO

def main():
    model = YOLO("runs/segment/runs/coffee-seg/weights/best.pt")

    metrics = model.val()

    print("📊 HASIL EVALUASI")
    print("mAP@0.5      :", metrics.box.map50)
    print("mAP@0.5:0.95 :", metrics.box.map)
    print("Precision    :", metrics.box.mp)
    print("Recall       :", metrics.box.mr)

if __name__ == "__main__":
    main()


Ultralytics 8.4.26  Python-3.14.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1660, 6144MiB)
YOLO11n-seg summary (fused): 114 layers, 2,836,908 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2.60.6 MB/s, size: 14.1 KB)
val: Scanning D:\dhiya_skirpsi\model_YOLO11-seg\dataset\valid\labels.cache... 2536 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2536/2536 590.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 159/159 4.2it/s 38.3s0.2ss
                   all       2536       3536      0.987       0.99       0.99      0.931      0.985      0.987      0.986      0.785
                broken        188        258      0.991      0.992      0.991      0.991      0.991      0.992      0.991      0.974
        foreign_matter        710       1424      0.951      0.895      0.957      0.821      0.919      0.865      0.908      0.577
  

In [5]:
# %%
from ultralytics import YOLO

model = YOLO("runs/segment/runs/coffee-seg/weights/best.pt")
model.info()

YOLO11n-seg summary: 204 layers, 2,844,948 parameters, 0 gradients, 9.7 GFLOPs


(204, 2844948, 0, 9.7384448)

In [3]:
#Hitung mIoU dan Dice
import os
import cv2
import numpy as np

img_dir = "dataset/valid/images"
label_dir = "dataset/valid/labels"

def polygon_to_mask(label_path, img_shape):
    mask = np.zeros(img_shape[:2], dtype=np.uint8)

    if not os.path.exists(label_path):
        return mask

    with open(label_path, "r") as f:
        for line in f:
            data = list(map(float, line.strip().split()))
            points = data[1:]

            poly = []
            for i in range(0, len(points), 2):
                x = int(points[i] * img_shape[1])
                y = int(points[i+1] * img_shape[0])
                poly.append([x, y])

            poly = np.array([poly], dtype=np.int32)
            cv2.fillPoly(mask, poly, 1)

    return mask


def compute_metrics(pred_mask, true_mask):
    pred = pred_mask.flatten()
    true = true_mask.flatten()

    intersection = np.sum(pred * true)
    union = np.sum(pred) + np.sum(true) - intersection

    iou = intersection / (union + 1e-6)
    dice = (2 * intersection) / (np.sum(pred) + np.sum(true) + 1e-6)

    return iou, dice


iou_list = []
dice_list = []

for i, r in enumerate(results):
    img_path = r.path
    img = cv2.imread(img_path)

    label_path = os.path.join(
        label_dir,
        os.path.basename(img_path).replace(".jpg", ".txt").replace(".png", ".txt")
    )

    true_mask = polygon_to_mask(label_path, img.shape)

    pred_mask = np.zeros(img.shape[:2], dtype=np.uint8)

    if r.masks is not None:
        for m in r.masks.data:
            mask = m.cpu().numpy()
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]))
            pred_mask = np.maximum(pred_mask, mask > 0.5)

    iou, dice = compute_metrics(pred_mask, true_mask)

    iou_list.append(iou)
    dice_list.append(dice)

print("Mean IoU  :", np.mean(iou_list))
print("Mean Dice :", np.mean(dice_list))

NameError: name 'results' is not defined